# Artist Performance Analytics


In [0]:
tracks = spark.table("spotify_silver.dim_tracks")
artists = spark.table("spotify_silver.dim_artists")
events = spark.table("spotify_silver.fact_listening_events")

In [0]:
tracks.createOrReplaceTempView("dim_tracks")
artists.createOrReplaceTempView("dim_artists")
events.createOrReplaceTempView("fact_listening_events")

In [0]:
%sql
CREATE OR REPLACE TABLE spotify_gold.artist_performance AS

SELECT
    a.artist_key,
    a.artist_name,

    COUNT(f.event_id) AS total_plays,

    COUNT(DISTINCT f.user_id) AS unique_listeners,

    ROUND(AVG(f.listen_duration_ms),2) AS avg_listen_duration_ms,

    ROUND(
        100.0 * AVG(CASE WHEN f.completed THEN 1 ELSE 0 END),
        2
    ) AS completion_rate,

    ROUND(
        100.0 * AVG(CASE WHEN f.skipped THEN 1 ELSE 0 END),
        2
    ) AS skip_rate

FROM fact_listening_events f

JOIN dim_tracks t
ON f.track_id = t.track_id

JOIN dim_artists a
ON t.artist_key = a.artist_key

GROUP BY
    a.artist_key,
    a.artist_name;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM spotify_gold.artist_performance
ORDER BY total_plays DESC
LIMIT 10;

artist_key,artist_name,total_plays,unique_listeners,avg_listen_duration_ms,completion_rate,skip_rate
2122,Taylor Swift,3829,2696,164503.57,80.49,20.06
2223,The Weeknd,1692,1439,160366.61,79.85,18.85
1239,Lana Del Rey,1168,1038,167785.05,80.05,19.95
146,Ariana Grande,1131,1026,163456.81,78.69,19.54
1602,Nirvana,1010,920,164113.75,78.32,16.63
1744,Post Malone,982,881,165374.38,81.16,19.35
598,Drake,933,867,168166.81,81.67,18.44
2183,The Neighbourhood,888,813,164036.58,81.87,18.58
1233,Lady Gaga,848,785,162202.47,80.31,20.87
1644,Olivia Rodrigo,813,753,167951.33,80.81,17.96


In [0]:
%sql
SELECT COUNT(*)
FROM spotify_gold.artist_performance;

COUNT(*)
2547


# Track Performance Analytics

In [0]:
%sql
CREATE OR REPLACE TABLE spotify_gold.track_performance AS

SELECT
    t.track_id,
    t.track_name,

    COUNT(f.event_id) AS total_plays,

    COUNT(DISTINCT f.user_id) AS unique_listeners,

    ROUND(AVG(f.listen_duration_ms),2) AS avg_listen_duration_ms,

    ROUND(
        100 * AVG(CASE WHEN f.completed THEN 1 ELSE 0 END),
        2
    ) AS completion_rate,

    ROUND(
        100 * AVG(CASE WHEN f.skipped THEN 1 ELSE 0 END),
        2
    ) AS skip_rate

FROM fact_listening_events f

JOIN dim_tracks t
ON f.track_id = t.track_id

GROUP BY
    t.track_id,
    t.track_name;

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from spotify_gold.track_performance
order by total_plays desc
limit 10;

track_id,track_name,total_plays,unique_listeners,avg_listen_duration_ms,completion_rate,skip_rate
776AftMmFFAWUIEAb3lHhw,ZITTI E BUONI,25,25,132539.56,72.0,4.0
52NGJPcLUzQq5w7uv4e5gf,Private Landing (feat. Justin Bieber & Future),25,25,172944.24,84.0,24.0
5ZdqVCiXrvMGpY8cux5g8t,Conceited,24,24,148255.88,66.67,41.67
3KkXRkHbMCARz0aVfEt68P,Sunflower - Spider-Man: Into the Spider-Verse,24,24,167670.42,83.33,12.5
5EpkyTNrxvsueT6z41bINu,"Sex, Drugs, Etc (Sped Up)",23,23,178172.52,91.3,43.48
2QpGZOhTCHHiKmpSO9FW4h,Follow God,23,23,167275.48,65.22,30.43
2l0xjJ6Fj72Bw4ReAJBrAo,The Ride of the Rohirrim,23,23,163621.22,73.91,13.04
5B3GZOZYXNzWpUXQC42hxZ,Sugar Talking,23,23,151138.61,78.26,17.39
69w5X6uTrOaWM32IetSzvO,Daydreaming,23,23,160241.17,73.91,17.39
3BUcoSTQjI1tJIKCKynOTt,Beggin',23,23,186596.61,91.3,17.39


In [0]:
%sql
select count(*) from spotify_gold.track_performance;

count(*)
8772


# Album Performance Analytics

In [0]:
%sql
CREATE OR REPLACE TABLE spotify_gold.album_performance AS

SELECT
    a.album_id,
    a.album_name,
    a.album_type,
    a.album_release_date,

    COUNT(f.event_id) AS total_plays,

    COUNT(DISTINCT f.user_id) AS unique_listeners,

    ROUND(AVG(f.listen_duration_ms), 2) AS avg_listen_duration_ms,

    ROUND(
        100 * AVG(CASE WHEN f.completed THEN 1 ELSE 0 END),
        2
    ) AS completion_rate,

    ROUND(
        100 * AVG(CASE WHEN f.skipped THEN 1 ELSE 0 END),
        2
    ) AS skip_rate

FROM fact_listening_events f

JOIN dim_tracks t
ON f.track_id = t.track_id

JOIN spotify_silver.dim_albums a
ON t.album_id = a.album_id

GROUP BY
    a.album_id,
    a.album_name,
    a.album_type,
    a.album_release_date;

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from spotify_gold.album_performance
order by total_plays desc
limit 10;

album_id,album_name,album_type,album_release_date,total_plays,unique_listeners,avg_listen_duration_ms,completion_rate,skip_rate
3FFGbUutKWN1c4f0CJR4Uh,Nevermind (Super Deluxe Edition),album,1991-09-26,750,702,166784.73,78.67,16.27
1MPAXuTVL2Ej5x0JHiSPq8,reputation Stadium Tour Surprise Song Playlist,album,2017-11-09,531,510,168745.49,82.49,16.01
5H7ixXZfsNMGbIE5OBSpcb,THE TORTURED POETS DEPARTMENT: THE ANTHOLOGY,album,2024-04-19,355,336,163821.14,77.75,16.34
2XGEyGU76kj55OdHWynX0S,Trilogy,album,2012-11-13,306,298,151213.64,76.47,19.61
43wFM1HquliY3iwKWzPN4y,Love Yourself 結 'Answer',album,2018-08-24,295,285,165567.5,79.66,18.31
2QRedhP5RmKJiJ1i8VgDGR,Whole Lotta Red,album,2020-12-25,289,285,167434.38,79.58,17.3
4hDok0OAJd57SGIT8xuWJH,Fearless (Taylor's Version),album,2021-04-09,279,272,165882.75,79.21,20.79
6rePArBMb5nLWEaY9aQqL4,The Fame Monster (Deluxe Edition),album,2009-11-05,275,268,161835.89,77.82,18.18
5BNUgsVK7qqfFnV046qHfW,Gloria (Special Edition),album,2023-04-12,268,264,166296.06,80.97,24.25
5VoeRuTrGhTbKelUfwymwu,Born To Die - The Paradise Edition,album,2012-01-01,268,258,165271.16,80.6,19.78


In [0]:
%sql
select count(*) from spotify_gold.album_performance

count(*)
5314


# User Engagement Analytics

In [0]:
%sql
CREATE OR REPLACE TABLE spotify_gold.user_engagement AS

SELECT
    u.user_id,
    u.country,
    u.subscription_type,
    u.preferred_device,
    u.age_group,
    u.signup_date,

    COUNT(f.event_id) AS total_plays,

    SUM(f.listen_duration_ms) AS total_listening_time_ms,

    ROUND(AVG(f.listen_duration_ms),2) AS avg_listening_duration_ms,

    ROUND(
        100 * AVG(CASE WHEN f.completed THEN 1 ELSE 0 END),
        2
    ) AS completion_rate,

    ROUND(
        100 * AVG(CASE WHEN f.skipped THEN 1 ELSE 0 END),
        2
    ) AS skip_rate

FROM spotify_silver.dim_users u

LEFT JOIN spotify_silver.fact_listening_events f
ON u.user_id = f.user_id

GROUP BY
    u.user_id,
    u.country,
    u.subscription_type,
    u.preferred_device,
    u.age_group,
    u.signup_date;

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from spotify_gold.user_engagement
order by total_plays desc
limit 10;

user_id,country,subscription_type,preferred_device,age_group,signup_date,total_plays,total_listening_time_ms,avg_listening_duration_ms,completion_rate,skip_rate
4397,Australia,Free,Mobile,45+,2022-10-31,39,7140300,183084.62,87.18,12.82
671,Brazil,Premium,Web,18-24,2022-12-18,39,6635454,170139.85,79.49,10.26
3267,India,Free,Mobile,35-44,2024-01-13,38,6770718,178176.79,71.05,13.16
1895,Germany,Free,Desktop,45+,2022-02-01,36,5832470,162013.06,83.33,22.22
102,UK,Premium,Mobile,18-24,2022-06-23,36,6542347,181731.86,80.56,22.22
4317,USA,Free,Mobile,18-24,2022-05-31,36,5419602,150544.5,83.33,19.44
1448,India,Free,Web,25-34,2023-08-17,35,6122052,174915.77,77.14,22.86
1440,UK,Premium,Mobile,25-34,2023-11-08,34,5816485,171073.09,70.59,32.35
1077,Germany,Free,Web,45+,2022-03-11,34,5776959,169910.56,58.82,14.71
1588,Australia,Free,Mobile,25-34,2023-09-09,34,4794567,141016.68,76.47,26.47


In [0]:
%sql
select count(*) from spotify_gold.user_engagement;

count(*)
5000
